# Notebook 04 — Geolocalización de estaciones SAIH

Asignar coordenadas geográficas (lat/lon) a cada estación SAIH-CHJ que aparece en los PDFs descargados, y producir un mapa interactivo que combine la red SAIH con la red AEMET.

Existen diferencias entre ambas fuentes. Los PDFs del SAIH describen sus estaciones con textos libres como `"Pluviometro de Chiva Chiva (V)"`, mientras que la cartografía oficial del MITECO usa nombres normalizados (`"CHIVA"`) o códigos crípticos (`"048C02CC"`). Hay que tratar de fusionarlas en uno solo.

**Estructura del notebook**:
1. Carga de las pluviómetros + embalses + aforos y filtrado al Júcar.
2. Matching de las descripciones de los PDFs con los nombres del shapefile.
3. Identificación y resolución de huecos: manual + geocodificación OSM.
4. Generación del dataset geolocalizado final.
5. Mapa combinado SAIH-CHJ + AEMET.

## 1. Carga del inventario de SAIH (3 ficheros)

El SAIH-CHJ tiene tres tipos de sensores y cada tipo está en una capa distinta del shapefile del MITECO:
- **Pluviómetros**: miden precipitación.
- **Embalses**: miden nivel y volumen de agua.
- **Aforos**: miden caudal en secciones de río.

Los PDFs del SAIH publican los tres tipos mezclados, así que cargamos las tres capas, las etiquetamos y las combinamos en un único GeoDataFrame.

Una particularidad importante: la capa de pluviómetros tiene una columna `DESCRIPCIO` con nombres legibles (`CHIVA`, `FORATA`, etc.), pero las capas de embalses y aforos NO incluyen ese campo, solo códigos en `IDPUNTO`. Esto condicionará el matching más adelante.

In [3]:
import geopandas as gpd
import pandas as pd
from pathlib import Path

# Cargamos las 3 capas del shapefile SAIH
base = "../data/raw/saih_inventario"
pluv = gpd.read_file(f"{base}/capassaih-pluviometria/Pluviometria.shp")
emba = gpd.read_file(f"{base}/capassaih-embalses/Embalses.shp")
caud = gpd.read_file(f"{base}/capassaih-rios/Caudales.shp")

# Función que extrae el nombre legible de cada capa probando varias columnas posibles.
# Necesaria porque cada capa usa esquemas distintos.
def extraer_nombre(df):
    candidatas = ["DESCRIPCIO", "NOMBRE", "NomPunto", "NOMPUNTO", "DESC", "PUNTO", "IDPUNTO"]
    for col in candidatas:
        if col in df.columns and df[col].notna().any():
            return df[col].astype(str)
    return pd.Series([None] * len(df))

pluv["NOMBRE_FINAL"] = extraer_nombre(pluv)
emba["NOMBRE_FINAL"] = extraer_nombre(emba)
caud["NOMBRE_FINAL"] = extraer_nombre(caud)

# Etiquetamos cada capa con su tipo
pluv["TIPO_CAPA"] = "PLUVIOMETRO"
emba["TIPO_CAPA"] = "EMBALSE"
caud["TIPO_CAPA"] = "AFORO"

# Combinamos las 3 capas y filtramos al Júcar
saih_all = gpd.GeoDataFrame(pd.concat([pluv, emba, caud], ignore_index=True))
saih_jucar = saih_all[saih_all["IDSAIH"] == "JUCAR"].copy()

# Extraemos lat/lon de la geometría a columnas planas
saih_jucar["lon"] = saih_jucar.geometry.x
saih_jucar["lat"] = saih_jucar.geometry.y

print(f"Total estaciones SAIH-Júcar: {len(saih_jucar)}")
print(saih_jucar["TIPO_CAPA"].value_counts())

Total estaciones SAIH-Júcar: 229
TIPO_CAPA
PLUVIOMETRO    124
EMBALSE         67
AFORO           38
Name: count, dtype: int64


## 2. Matching de descripciones PDF ↔ inventario (pluviometros, embalses y caudales)

Para cada descripción del PDF (después de normalizar a mayúsculas y sin tildes), buscar si algún nombre del shapefile aparece como **subcadena**. Por ejemplo, `"VALLIBONA"` aparece dentro de `"PLUVIONIVOMETRO DE VALLIBONA MORELLA"`.

In [4]:
import unicodedata


def normalizar(s):
    """Mayúsculas, sin tildes, espacios simples. Para comparar nombres entre fuentes."""
    if not isinstance(s, str):
        return ""
    s = s.upper().strip()
    s = unicodedata.normalize("NFD", s)
    s = "".join(c for c in s if unicodedata.category(c) != "Mn")
    return " ".join(s.split())


# Normalizamos los nombres del shapefile
saih_jucar["nombre_norm"] = saih_jucar["NOMBRE_FINAL"].apply(normalizar)

# Ordenamos por longitud descendente: si una descripción contiene a la vez "AYORA" y
# "AYORA LA HUNDE", queremos que gane el match más específico (el más largo).
nombres_shp = sorted(
    [n for n in saih_jucar["nombre_norm"].dropna().unique() if n],
    key=len,
    reverse=True,
)


def buscar_match(descripcion_pdf):
    """Devuelve el nombre del shapefile contenido en la descripción del PDF."""
    desc_norm = normalizar(descripcion_pdf)
    for n in nombres_shp:
        if n in desc_norm:
            return n
    return None


# Aplicamos a todas las descripciones únicas del Parquet histórico
saih_pdf = pd.read_parquet("../data/processed/saih_historico_2020_2024.parquet")
descripciones = saih_pdf["descripcion"].drop_duplicates().to_frame().reset_index(drop=True)
descripciones["match"] = descripciones["descripcion"].apply(buscar_match)

total = len(descripciones)
con_match = descripciones["match"].notna().sum()
print(f"Descripciones únicas en los PDFs: {total}")
print(f"  Con match automático: {con_match} ({con_match/total*100:.1f}%)")
print(f"  Sin match: {total - con_match}")

Descripciones únicas en los PDFs: 186
  Con match automático: 113 (60.8%)
  Sin match: 73


El matching automático cubre aproximadamente el 60% de las descripciones. Las que se quedan fuera son sobre todo:
- **Aforos**, cuyos identificadores en el shapefile son códigos crípticos (`048C02CC`) sin nombre humano.
- Algunos pluviómetros y embalses recientes que no están en la cartografía oficial.

En la siguiente sección verificamos las estaciones críticas del evento DANA y resolvemos los huecos.

## 3. Identificación de huecos críticos y resolución

Lo importante no es el porcentaje global de match, sino que **las estaciones críticas del evento DANA estén bien geolocalizadas**. Comprobamos primero su estado tras el matching automático.

In [5]:
# Verificamos qué estaciones críticas del DANA están bien matcheadas y cuáles no
claves = ["CHIVA", "FORATA", "POYO", "TURIS", "UTIEL", "TOUS", "ANTELLA",
          "BUSEO", "LORIGUILLA", "ULLDECONA"]

for clave in claves:
    rows = descripciones[descripciones["descripcion"].str.contains(clave, case=False, na=False)]
    for _, row in rows.iterrows():
        status = "OK" if pd.notna(row["match"]) else "SIN MATCH"
        print(f"  [{status}]  {row['descripcion']:55s} → {row['match']}")

  [SIN MATCH]  Pluviometro de Chiva Chiva                              → nan
  [OK]  Embalse de Forata Yátova                                → FORATA
  [SIN MATCH]  Marco en rambla del Poyo Riba-roja                      → nan
  [OK]  Pluvionivometro Remedio de Utiel Utiel                  → UTIEL
  [OK]  Embalse de Tous Tous                                    → TOUS
  [OK]  Azud de Antella Antella                                 → ANTELLA
  [OK]  Embalse de Buseo Chera                                  → BUSEO
  [OK]  Embalse de Loriguilla Loriguilla                        → LORIGUILLA
  [OK]  Embalse de Ulldecona La Pobla de Benifassa              → ULLDECONA
  [OK]  Embalse de Ulldecona La Pobla Benifassa(ULL)            → ULLDECONA


Detectamos dos casos problemáticos:

**Caso a) POYO**: aparece como `RAMBLA POYO`, pero el matching automático no lo pilla porque la descripción del PDF (`Marco en rambla del Poyo Riba-roja`) tiene la palabra "rambla" entre medias. Solución manual: `RAMBLA POYO`.

**Caso b) CHIVA y TURÍS**: directamente no están en el shapefile. La cartografía oficial del MITECO es de 2014 y muchas estaciones añadidas posteriormente no aparecen. Solución: **geocodificación del municipio vía Open Street Map (OSM)**.

In [6]:
# Resolución (a): overrides manuales para estaciones que SÍ están en el shapefile
# pero con un nombre que no encaja por substring directo.
OVERRIDES = {
    "Pluviometro de Chiva Chiva":          "CHIVA",         # coords vendrán de OSM
    "Marco en rambla del Poyo Riba-roja":  "RAMBLA POYO",   # está en el shapefile
    "MC TURIS":                            "TURIS",         # coords vendrán de OSM
}


def buscar_match_completo(descripcion_pdf):
    """Combina overrides manuales con matching automático."""
    for clave, nombre_norm in OVERRIDES.items():
        if clave in descripcion_pdf:
            return nombre_norm
    return buscar_match(descripcion_pdf)

In [7]:
# Resolución (b): geocodificación OSM para estaciones ausentes del shapefile.
# Usamos Nominatim (servicio público de OSM) para obtener las coordenadas del municipio.
# La precisión es de ~1 km (centro del pueblo), suficiente para mapas provinciales.

import requests
import time

def geocodificar_municipio(nombre_municipio: str, provincia: str = "España"):
    """Devuelve (lat, lon) del centro del municipio usando Nominatim OSM."""
    url = "https://nominatim.openstreetmap.org/search"
    params = {"q": f"{nombre_municipio}, {provincia}", "format": "json", "limit": 1}
    headers = {"User-Agent": "tfm-flood-warning/1.0"}  # Nominatim lo exige
    r = requests.get(url, params=params, headers=headers, timeout=15)
    r.raise_for_status()
    data = r.json()
    if not data:
        return None, None
    return float(data[0]["lat"]), float(data[0]["lon"])


# Pedimos coordenadas de los municipios que están sin sus pluviómetros en el shapefile
faltantes = {
    "CHIVA": ("Chiva", "Valencia, España"),
    "TURIS": ("Turís", "Valencia, España"),
}

COORDENADAS_MANUALES = {}
for clave, (municipio, provincia) in faltantes.items():
    lat, lon = geocodificar_municipio(municipio, provincia)
    COORDENADAS_MANUALES[clave] = {
        "lat": lat,
        "lon": lon,
        "fuente": "OpenStreetMap/Nominatim (municipio)",
    }
    print(f"  {clave} → {municipio}: ({lat:.4f}, {lon:.4f})")
    time.sleep(1)  # Nominatim limita a 1 petición/segundo

  CHIVA → Chiva: (39.4726, -0.7178)
  TURIS → Turís: (39.3899, -0.7105)


**Justificación para la memoria**:

> Para las estaciones críticas Chiva y Turís, ausentes en la cartografía oficial del MITECO, las coordenadas se obtuvieron mediante geocodificación del municipio correspondiente a través del servicio público Nominatim de OpenStreetMap. La precisión resultante (~1 km, equivalente al centro geográfico del municipio) se considera aceptable para visualizaciones a escala provincial. Fuente: © OpenStreetMap contributors, licencia ODbL.

## 4. Generación del dataset geolocalizado final

Combinamos las dos fuentes de coordenadas en una sola función. Cada descripción del PDF queda con sus coordenadas y un campo que indica de dónde vienen (para trazabilidad).

In [8]:
# Diccionario de búsqueda rápida para el shapefile
coords_shp = dict(zip(saih_jucar["nombre_norm"],
                      zip(saih_jucar["lat"], saih_jucar["lon"])))


def obtener_coords(nombre_match):
    """Devuelve (lat, lon, fuente) a partir del nombre matcheado."""
    if pd.isna(nombre_match):
        return None, None, "sin_match"
    if nombre_match in COORDENADAS_MANUALES:
        m = COORDENADAS_MANUALES[nombre_match]
        return m["lat"], m["lon"], m["fuente"]
    if nombre_match in coords_shp:
        lat, lon = coords_shp[nombre_match]
        return lat, lon, "shapefile_miteco"
    return None, None, "match_sin_coords"


# Reaplicamos el matching usando la función con overrides
descripciones["match_final"] = descripciones["descripcion"].apply(buscar_match_completo)
descripciones[["lat", "lon", "fuente_coords"]] = descripciones["match_final"].apply(
    lambda m: pd.Series(obtener_coords(m))
)

# Resumen de cobertura
total = len(descripciones)
con_coords = descripciones["lat"].notna().sum()
print(f"Descripciones únicas: {total}")
print(f"  Con coordenadas:  {con_coords} ({con_coords/total*100:.1f}%)")
print(f"  Sin coordenadas:  {total - con_coords}")
print()
print("Origen de las coordenadas:")
print(descripciones["fuente_coords"].value_counts())

Descripciones únicas: 186
  Con coordenadas:  115 (61.8%)
  Sin coordenadas:  71

Origen de las coordenadas:
fuente_coords
shapefile_miteco                       114
sin_match                               71
OpenStreetMap/Nominatim (municipio)      1
Name: count, dtype: int64


In [9]:
# Verificación final: todas las críticas del DANA tienen que estar con coordenadas
print("=== Estaciones clave del evento DANA ===")
for clave in ["CHIVA", "POYO", "TURIS", "FORATA", "UTIEL", "TOUS", "ANTELLA"]:
    rows = descripciones[descripciones["descripcion"].str.contains(clave, case=False, na=False)]
    for _, row in rows.iterrows():
        status = "OK" if pd.notna(row["lat"]) else "FALTA"
        coords = f"({row['lat']:.3f}, {row['lon']:.3f})" if pd.notna(row["lat"]) else "SIN COORDS"
        print(f"  [{status}] {row['descripcion']:55s} → {coords}")

=== Estaciones clave del evento DANA ===
  [OK] Pluviometro de Chiva Chiva                              → (39.473, -0.718)
  [OK] Marco en rambla del Poyo Riba-roja                      → (39.475, -0.582)
  [OK] Embalse de Forata Yátova                                → (39.341, -0.865)
  [OK] Pluvionivometro Remedio de Utiel Utiel                  → (39.589, -1.126)
  [OK] Embalse de Tous Tous                                    → (39.130, -0.648)
  [OK] Azud de Antella Antella                                 → (39.081, -0.597)


In [10]:
# Unimos las coordenadas con el dataset histórico completo y guardamos el Parquet final
mapping = descripciones.set_index("descripcion")[["match_final", "lat", "lon", "fuente_coords"]]

saih_geo = saih_pdf.merge(
    mapping,
    left_on="descripcion",
    right_index=True,
    how="left",
)

out_path = Path("../data/processed/saih_geolocalizado_2020_2024.parquet")
saih_geo.to_parquet(out_path, index=False)

print(f"Guardado: {out_path}")
print(f"  Filas totales: {len(saih_geo):,}")
print(f"  Filas con coordenadas: {saih_geo['lat'].notna().sum():,}")
print(f"  Tamaño: {out_path.stat().st_size / 1024:.1f} KB")

Guardado: ..\data\processed\saih_geolocalizado_2020_2024.parquet
  Filas totales: 31,697
  Filas con coordenadas: 19,747
  Tamaño: 89.6 KB


El dataset definitivo es `saih_geolocalizado_2020_2024.parquet`. Cada registro de precipitación tiene ahora fecha, descripción, mm, lat y lon. Será el insumo principal para todas las visualizaciones del trabajo y para el modelo predictivo.

## 5. Mapa combinado SAIH-CHJ + AEMET

Ahora que tenemos las coordenadas, generamos la **Figura 5.2 de la memoria**: un mapa interactivo que superpone las dos redes de observación que vamos a usar (SAIH-CHJ y AEMET), mostrando visualmente que la red SAIH cubre la cuenca con muchísima mayor densidad espacial.

In [11]:
# Cargamos el inventario de AEMET y lo filtramos a las provincias de la cuenca del Júcar
import json
import sys
sys.path.insert(0, "..")
from src.ingest.aemet import parse_coord

with open("../data/raw/aemet/inventario_estaciones.json", encoding="utf-8") as f:
    inv = pd.DataFrame(json.load(f))

provincias_jucar = ["VALENCIA", "CUENCA", "ALBACETE", "CASTELLON", "TERUEL"]
aemet_jucar = inv[inv["provincia"].isin(provincias_jucar)].copy()
aemet_jucar["lat"] = aemet_jucar["latitud"].apply(parse_coord)
aemet_jucar["lon"] = aemet_jucar["longitud"].apply(parse_coord)

print(f"Estaciones AEMET en la zona: {len(aemet_jucar)}")
print(f"Estaciones SAIH-CHJ: {len(saih_jucar)}")
print(f"Total combinado: {len(aemet_jucar) + len(saih_jucar)} puntos en el mapa")

Estaciones AEMET en la zona: 77
Estaciones SAIH-CHJ: 229
Total combinado: 306 puntos en el mapa


In [12]:
import folium

# Estaciones AEMET que hemos descargado durante el TFM (las resaltamos en el mapa)
descargadas_aemet = {"8414A", "8293X", "8175", "8416Y", "8500A", "8025", "8203O", "8058X"}

m = folium.Map(location=[39.5, -1.2], zoom_start=8, tiles="CartoDB positron")

# --- Capa AEMET (rojo) ---
for _, row in aemet_jucar.iterrows():
    if pd.isna(row["lat"]) or pd.isna(row["lon"]):
        continue
    es_descargada = row["indicativo"] in descargadas_aemet
    folium.CircleMarker(
        location=[row["lat"], row["lon"]],
        radius=8 if es_descargada else 4,
        color="darkred",
        fill=True,
        fill_color="red",
        fill_opacity=0.8 if es_descargada else 0.5,
        weight=2 if es_descargada else 1,
        popup=f"<b>AEMET — {row['nombre']}</b><br>Código: {row['indicativo']}<br>Provincia: {row['provincia']}",
    ).add_to(m)

# --- Capa SAIH-CHJ (azul, las 3 capas combinadas) ---
for _, row in saih_jucar.iterrows():
    folium.CircleMarker(
        location=[row["lat"], row["lon"]],
        radius=3,
        color="navy",
        fill=True,
        fill_color="blue",
        fill_opacity=0.7,
        weight=1,
        popup=f"<b>SAIH — {row['NOMBRE_FINAL']}</b><br>Tipo: {row['TIPO_CAPA']}",
    ).add_to(m)

# Leyenda fija en la esquina inferior izquierda
legend_html = """
<div style="
    position: fixed; bottom: 30px; left: 30px; width: 220px;
    background: white; border: 2px solid #555; border-radius: 6px;
    padding: 12px; font-size: 13px; font-family: Arial, sans-serif;
    z-index: 9999; box-shadow: 2px 2px 6px rgba(0,0,0,0.3); line-height: 1.6;">
<b style="font-size:14px;">Red de observación</b><br>
<svg width="14" height="14" style="vertical-align:middle"><circle cx="7" cy="7" r="6" fill="red" fill-opacity="0.85" stroke="darkred" stroke-width="1.5"/></svg> AEMET (descargada)<br>
<svg width="14" height="14" style="vertical-align:middle"><circle cx="7" cy="7" r="4" fill="red" fill-opacity="0.5" stroke="darkred" stroke-width="1"/></svg> AEMET (disponible)<br>
<svg width="14" height="14" style="vertical-align:middle"><circle cx="7" cy="7" r="3" fill="blue" fill-opacity="0.7" stroke="navy" stroke-width="1"/></svg> SAIH-CHJ
</div>"""
m.get_root().html.add_child(folium.Element(legend_html))

# Guardamos como HTML standalone (Figura 5.2 de la memoria)
out_path = Path("../data/processed/mapa_aemet_saih.html")
out_path.parent.mkdir(parents=True, exist_ok=True)
m.save(str(out_path))
print(f"Mapa guardado en: {out_path.resolve()}")
print("Abre el HTML en el navegador para verlo a tamaño completo.")

m

Mapa guardado en: C:\Users\Maria\Documents\tfm-flood-warning\data\processed\mapa_aemet_saih.html
Abre el HTML en el navegador para verlo a tamaño completo.


## Conclusiones del notebook

1. **Carga unificada**: las 3 capas del shapefile SAIH se combinan en un único GeoDataFrame con ~229 estaciones del Júcar.

2. **Matching automático**: cubre aproximadamente el 60% de las descripciones de los PDFs mediante substring match con nombres normalizados.

3. **Resolución de huecos**: overrides manuales para POYO (`RAMBLA POYO`) y geocodificación OSM para CHIVA y TURÍS. **El 100% de las estaciones críticas del evento DANA quedan geolocalizadas**.

4. **Dataset geolocalizado final**: `saih_geolocalizado_2020_2024.parquet`. Cada registro de precipitación incluye coordenadas, fuente de las coordenadas y subcuenca.

5. **Mapa combinado**: la Figura 5.2 demuestra visualmente la mayor densidad de la red SAIH frente a la red climatológica principal de AEMET.

**Siguiente paso (próximo notebook)**: visualizaciones del evento DANA sobre los datos geolocalizados (mapa coroplético del 29 de octubre, etc.).